# Frankenstein.jl: The Intelligent Meta-Solver

Welcome to the **Frankenstein.jl** showcase! This notebook demonstrates the power of a modern, adaptive meta-solver. Frankenstein isn't just an algorithm; it's a **brain** that analyzes your physics and chooses the best numerical tools for the job.

## Environment Setup

In [3]:
using Pkg
Pkg.activate(".") 
Pkg.instantiate()

using Frankenstein
using Plots, LinearAlgebra, SparseArrays, SciMLBase
import YAML

include("test/combustion/Chemistry.jl")

println("It's Alive!")

  Activating project at `c:\Users\jelte\projects\Frankenstein`


It's Alive!


# Part 1: The Intelligence Gauntlet

In this section, we subject Frankenstein to a series of benchmarks that test its ability to identify **Density**, **Sparsity**, and **Scale**.

In [ ]:
function laplacian_2d_sparsity(N)
    I_idx, J_idx = Int[], Int[]
    for j in 1:N, i in 1:N
        idx = (j-1)*N + i
        push!(I_idx, idx); push!(J_idx, idx)
        if i > 1; push!(I_idx, idx); push!(J_idx, idx-1); end
        if i < N; push!(I_idx, idx); push!(J_idx, idx+1); end
        if j > 1; push!(I_idx, idx); push!(J_idx, idx-N); end
        if j < N; push!(I_idx, idx); push!(J_idx, idx+N); end
    end
    return sparse(I_idx, J_idx, ones(length(I_idx)))
end

println("--- Benchmark 1: The Oregonator (Small & Stiff) ---")
let 
    function oregonator!(du, u, p, t)
        s, w, q = 77.27, 0.161, 8.375e-6
        du[1] = s*(u[2] + u[1]*(1 - q*u[1] - u[2]))
        du[2] = (u[3] - (1 + u[1])*u[2]) / s
        du[3] = w*(u[1] - u[3])
    end
    global prob_oregon = ODEProblem(oregonator!, [1.0, 2.0, 3.0], (0.0, 2.0))
end
sol_oregon = solve(prob_oregon, Monster())

println("\n--- Benchmark 2: Dense Kuramoto Model (100% Dense, n=100) ---")
let N=100, K=5.0
    ω = rand(N)
    function kuramoto!(du, u, p, t)
        for i in 1:N
            sum_terms = 0.0
            for j in 1:N; sum_terms += sin(u[j] - u[i]); end
            du[i] = ω[i] + (K/N) * sum_terms
        end
    end
    global prob_kura = ODEProblem(kuramoto!, rand(N), (0.0, 5.0))
end
sol_kura = solve(prob_kura, Monster())

println("\n--- Benchmark 3: 2D Heat Equation (Ultra-Sparse, <1% Density, n=900) ---")
let N=30, dx=1.0/(30+1), c=0.1/dx^2
    function heat!(du,u,p,t)
        for j in 1:N, i in 1:N
            idx=Int((j-1)*N+i)
            l = i > 1 ? u[idx-1] : 0.0; r = i < N ? u[idx+1] : 0.0
            d = j > 1 ? u[idx-N] : 0.0; u_p = j < N ? u[idx+N] : 0.0
            du[idx]=c*(l+r+d+u_p-4*u[idx])
        end
    end
    global prob_heat = ODEProblem(ODEFunction(heat!, jac_prototype=laplacian_2d_sparsity(N)), rand(N^2), (0.0, 0.1))
end
sol_heat = solve(prob_heat, Monster())

p_o=plot(sol_oregon, title="Oregonator (Symbolics)")
p_k=plot(sol_kura, title="Dense Kuramoto (ForwardDiff)", legend=false)
p_h=plot(reshape(sol_heat[end], 30, 30), st=:surface, title="Sparse Heat (Sparse-AD)", legend=false)
plot(p_o, p_k, p_h, layout=(3,1), size=(800, 900))

LoadError: ParseError:
[90m# Error @ [0;0m]8;;file://c:/Users/jelte/projects/Frankenstein/In[6]#12:20\[90mIn[6]:12:20[0;0m]8;;\
    return sparse(I_idx, J_idx, ones)
    (length(I_idx))[48;2;120;70;70m[0;0m)
[90m#                  └ ── [0;0m[91mExpected `end`[0;0m

# Part 2: The Ultimate Challenge - High-Fidelity Combustion

Now we challenge Frankenstein with a **1D reactive-diffusive PDE**. This system is not just stiff; it is multiscale, sparse, and extremely sensitive to ignition.

### Setting up the Chemical Field and PDE Operator
We load a 30-species mechanism and configure a sparse Jacobian operator for maximum performance.

In [ ]:
mech_path = "test/combustion/LuMechanism.yaml"
mechanism_data = load_mechanism_data(mech_path)
species_list, reaction_list, species_index_map = process_chemical_data(mechanism_data)
kinetics_data = build_kinetics_list(reaction_list)
num_species = length(species_list); num_vars = num_species + 1
L = 0.02; nx = 30; dx = L / (nx - 1); P = 101325.0; R_gas = 8.314
S = build_stoichiometric_matrix(reaction_list, num_species)

function combustion_pde!(du, u, p, t)
    u_mat = reshape(u, num_vars, nx); du_mat = reshape(du, num_vars, nx)
    fill!(du, 0.0); r_buffer = zeros(eltype(u), length(reaction_list))
    for j in 2:(nx-1)
        T = u_mat[1, j]
        compute_reaction_rates!(r_buffer, @view(u_mat[1:end, j]), kinetics_data, species_list)
        reaction_source = S * r_buffer
        q_dot = 0.0; total_cv = 0.0
        for i in 1:num_species
            h = h0(T, species_list[i].thermo) - R_gas * T
            cv = species_cp(T, species_list[i].thermo) - R_gas
            q_dot -= h * reaction_source[i]
            total_cv += u_mat[i+1, j] * cv
        end
        dTdx2 = (u_mat[1, j+1] - 2*u_mat[1, j] + u_mat[1, j-1]) / dx^2
        kappa = conductivity(T)
        du_mat[1, j] = (kappa * dTdx2 + q_dot) / max(total_cv, 1.0)
        for i in 1:num_species
            D_i = 5e-5; dCdx2 = (u_mat[i+1, j+1] - 2*u_mat[i+1, j] + u_mat[i+1, j-1]) / dx^2
            du_mat[i+1, j] = D_i * dCdx2 + reaction_source[i]
        end
    end
    du_mat[:, nx] .= (u_mat[:, nx] .- u_mat[:, nx-1]) / dx 
end

I_idx, J_idx = Int[], Int[]
for j in 1:nx
    vars_j = ((j-1)*num_vars+1):(j*num_vars)
    for v1 in vars_j, v2 in vars_j; push!(I_idx, v1); push!(J_idx, v2); end
    if j > 1; vars_prev = ((j-2)*num_vars+1):((j-1)*num_vars); for i in 1:num_vars; push!(I_idx, vars_j[i]); push!(J_idx, vars_prev[i]); end; end
    if j < nx; vars_next = (j*num_vars+1):((j+1)*num_vars); for i in 1:num_vars; push!(I_idx, vars_j[i]); push!(J_idx, vars_next[i]); end; end
end
jac_proto = sparse(I_idx, J_idx, ones(length(I_idx)))
ff = ODEFunction(combustion_pde!, jac_prototype=jac_proto)

u0 = zeros(num_vars * nx); u0_mat = reshape(u0, num_vars, nx)
cfg_inlet = ChemistryConfig(temperature=300.0, pressure=P, fuel_mixture=Dict("CH4"=>0.05, "H2"=>0.04), air_percentage=0.91)
c_inlet = initialize_concentrations(cfg_inlet, species_list, species_index_map)
for j in 1:nx; u0_mat[:, j] .= c_inlet; end
spark_center = Int(nx ÷ 2)
for j in (spark_center-3):(spark_center+3)
    u0_mat[1, j] = 2500.0
    for rad in ["OH", "O", "H", "HO2", "CH3"]
        if haskey(species_index_map, rad); u0_mat[species_index_map[rad]+1, j] += 0.1; end
    end
end
prob_combustion = ODEProblem(ff, u0, (0.0, 1e-3))

println("Combustion Physics Locked and Loaded. n = $(length(u0)) variables.")

Combustion Physics Locked and Loaded. n = 930 variables.


: 

### Solving the Ultimate Challenge
Watch how Frankenstein chooses the **Sparse ForwardDiff** backend for this huge system and adapts to the initial stiffness surge.

In [ ]:
let 
    print_step = 20
    # Using a local counter inside a let block to avoid DEStats field-naming conflicts
    step_counter = 0
    cb = DiscreteCallback((u,t,integrator)->true, integrator -> begin
        step_counter += 1
        if step_counter % print_step == 0
            u_mat = reshape(integrator.u, num_vars, nx)
            @info "[Step $step_counter] t=$(round(integrator.t, sigdigits=4))s | dt=$(round(integrator.dt, sigdigits=4))s | Max T=$(round(maximum(u_mat[1, :]), digits=1))K"
        end
    end)

    global sol_combustion = solve(prob_combustion, Monster(), reltol=1e-6, abstol=1e-9, callback=cb)
end

println("\n--- Combustion Solve Complete ---")

[ Info: [Frankenstein Analysis] System Size: 930 | Sparse: true | Density: 3.54%
[ Info: [Frankenstein] Synchronized detected sparsity pattern and disabled FunctionWrappers.
[ Info: [Frankenstein] Initializing with Sundials.CVODE_BDF{:Newton, :Dense, Nothing, Nothing}
[ Info: [Frankenstein] Backend selection: Backend: ADTypes.AutoFiniteDiff{Val{:forward}, Val{:forward}, Val{:hcentral}, Nothing, Nothing, Bool}


: 

### The Resulting Flame Structure
We visualize the temperature wave and the evolution of the main fuel species.

In [ ]:
x_coords = collect(range(0, L, length=nx))
final_u = reshape(sol_combustion.u[end], num_vars, nx)
p1 = plot(x_coords, final_u[1, :], lw=2, title="Temperature Wave (K)", label="T", color=:red, ylabel="T [K]")
p2 = plot(title="Species Concentration Structure", xlabel="Position (m)", ylabel="C [mol/m³]")
for spec in ["H2", "CH4", "CO2", "OH"]
    if haskey(species_index_map, spec)
        plot!(p2, x_coords, final_u[species_index_map[spec]+1, :], label=spec, lw=2)
    end
end
plot(p1, p2, layout=(2,1), size=(800, 700))